# Amazon DynamoDB — Day-to-Day Operations with Python

**VidyaSankalp · AWS Series · Module 4 notebook** · Tutor: Prudhvi Akella

```
 1 Setup → 2 Create table → 3 Add items → 4 Get one → 5 Scan all → 6 Update → 7 Delete one
        → 8 Index (GSI) → 9 Chat history table → 10 Chat memory → 11 Clean up
```

Same data as the PDF: AcmeCorp's HR assistant keeps **employees** and **chat messages** in DynamoDB.

**Before you start:** run `aws configure` once (Module 1, "Before you start"). Run the cells top to bottom.

## 1. Setup

```
 notebook ──> boto3.resource("dynamodb") ──> your AWS login ──> DynamoDB in us-east-1
```

We use `boto3.resource`, which takes **normal Python values** — no `{"S": "..."}` wrapping like the CLI needs.

In [ ]:
# %pip install boto3 pandas      # run once if not installed

import time

import boto3
import pandas as pd
from boto3.dynamodb.conditions import Key

REGION = "us-east-1"
dynamodb = boto3.resource("dynamodb", region_name=REGION)

## 2. Create a table

```
 create_table(name, key, on-demand) ──> wait until ACTIVE ──> ready
```

Only the **key** is declared. Every other attribute (`name`, `city` …) arrives with the items.

In [ ]:
employees = dynamodb.create_table(
    TableName="Employees",
    KeySchema=[{"AttributeName": "emp_id", "KeyType": "HASH"}],              # HASH = partition key
    AttributeDefinitions=[{"AttributeName": "emp_id", "AttributeType": "S"}], # S = string
    BillingMode="PAY_PER_REQUEST",                                           # on-demand
)
employees.wait_until_exists()
print("Status:", employees.table_status)

> Ran this before and the table already exists? Use it instead of creating it:
> `employees = dynamodb.Table("Employees")`

## 3. Add items

```
 one item        put_item(Item={...})
 many items      batch_writer() — sends them in groups for you
```

One item:

In [ ]:
employees.put_item(Item={
    "emp_id": "E1001",
    "name": "Anita",
    "department": "HR",
    "city": "Hyderabad",
    "leave_balance": 18,
})
print("Added E1001")

Many items — notice Sana has a `skills` attribute the others do not:

In [ ]:
people = [
    {"emp_id": "E1002", "name": "Ravi", "department": "Finance", "city": "Mumbai", "leave_balance": 12},
    {"emp_id": "E1003", "name": "Sana", "department": "IT", "city": "Pune", "leave_balance": 20,
     "skills": ["python", "aws"]},
    {"emp_id": "E1004", "name": "Kiran", "department": "HR", "city": "Chennai", "leave_balance": 15},
]

with employees.batch_writer() as batch:
    for person in people:
        batch.put_item(Item=person)

print("Added", len(people), "employees")

> ⚠️ `put_item` with a key that already exists **replaces the whole item**. Use `update_item` (section 6) to change a few fields.

## 4. Get one item by key

```
 get_item(Key={"emp_id": "E1001"}) ──> response["Item"]   (no "Item" if the key does not exist)
```

In [ ]:
response = employees.get_item(Key={"emp_id": "E1001"})
anita = response["Item"]
print(anita)
print("Leave balance:", anita["leave_balance"])

In [ ]:
response = employees.get_item(Key={"emp_id": "E9999"})
print("Found?", "Item" in response)

> Numbers come back as `Decimal('18')`, not `18`. That is how boto3 keeps numbers exact.
> Use `int(...)` or `float(...)` when you need a normal number. To **write** a decimal number, use `from decimal import Decimal` and pass `Decimal("4.5")`, not `4.5`.

In [ ]:
print(type(anita["leave_balance"]), int(anita["leave_balance"]))

## 5. Read all items (scan)

```
 scan() ──> reads the WHOLE table
```

Fine for a small table while learning. In real apps, use `get_item` or `query` instead.

In [ ]:
items = employees.scan()["Items"]
pd.DataFrame(items)

## 6. Update an item

```
 update_item(Key, UpdateExpression) ──> changes only the listed attributes
```

In [ ]:
response = employees.update_item(
    Key={"emp_id": "E1001"},
    UpdateExpression="SET city = :c",
    ExpressionAttributeValues={":c": "Bengaluru"},
    ReturnValues="ALL_NEW",                         # give back the item after the change
)
print(response["Attributes"])

### Safe update with a condition

Take leave days **only if** enough are left. If not, DynamoDB refuses and nothing changes.

```
 SET leave_balance = leave_balance - :days   IF leave_balance >= :days
```

In [ ]:
def take_leave(emp_id, days):
    try:
        response = employees.update_item(
            Key={"emp_id": emp_id},
            UpdateExpression="SET leave_balance = leave_balance - :days",
            ConditionExpression="leave_balance >= :days",
            ExpressionAttributeValues={":days": days},
            ReturnValues="UPDATED_NEW",
        )
        print(f"{emp_id}: approved, balance now {response['Attributes']['leave_balance']}")
    except dynamodb.meta.client.exceptions.ConditionalCheckFailedException:
        print(f"{emp_id}: refused, not enough leave")

take_leave("E1001", 2)      # 18 -> 16
take_leave("E1001", 100)    # refused

## 7. Delete an item

```
 delete_item(Key={"emp_id": ...})
```

In [ ]:
employees.delete_item(Key={"emp_id": "E1004"})
print([item["emp_id"] for item in employees.scan()["Items"]])

## 8. Find by another attribute: Global Secondary Index (GSI)

The table can only be looked up by `emp_id`. To ask "who is in HR?", add a GSI with key `department`.

```
 update table: add GSI department-index ──> wait until ACTIVE ──> query(IndexName=..., department = "HR")
```

Like `CREATE INDEX ... ON employees (department)` in SQL — but you must **name the index** in the query.

In [ ]:
employees.update(
    AttributeDefinitions=[{"AttributeName": "department", "AttributeType": "S"}],
    GlobalSecondaryIndexUpdates=[{"Create": {
        "IndexName": "department-index",
        "KeySchema": [{"AttributeName": "department", "KeyType": "HASH"}],
        "Projection": {"ProjectionType": "ALL"},          # copy all attributes into the index
    }}],
)

# wait until the index is ready (about a minute on AWS)
while True:
    employees.reload()
    status = employees.global_secondary_indexes[0]["IndexStatus"]
    print("Index status:", status)
    if status == "ACTIVE":
        break
    time.sleep(10)

In [ ]:
response = employees.query(
    IndexName="department-index",
    KeyConditionExpression=Key("department").eq("HR"),
)
for item in response["Items"]:
    print(item["emp_id"], item["name"])

## 9. Chat history table (partition key + sort key + TTL)

```
 ChatMessages
   partition key  session_id   →  groups one conversation
   sort key       created_at   →  keeps its messages in time order
   TTL            expires_at   →  DynamoDB deletes old messages for free
```

In [ ]:
chats = dynamodb.create_table(
    TableName="ChatMessages",
    KeySchema=[
        {"AttributeName": "session_id", "KeyType": "HASH"},    # partition key
        {"AttributeName": "created_at", "KeyType": "RANGE"},   # sort key
    ],
    AttributeDefinitions=[
        {"AttributeName": "session_id", "AttributeType": "S"},
        {"AttributeName": "created_at", "AttributeType": "S"},
    ],
    BillingMode="PAY_PER_REQUEST",
)
chats.wait_until_exists()

dynamodb.meta.client.update_time_to_live(
    TableName="ChatMessages",
    TimeToLiveSpecification={"Enabled": True, "AttributeName": "expires_at"},
)
print("ChatMessages ready, TTL on expires_at")

A small function to save one message. It keeps the message for 30 days:

In [ ]:
from datetime import datetime, timezone

def save_message(session_id, role, text):
    now = datetime.now(timezone.utc)
    chats.put_item(Item={
        "session_id": session_id,
        "created_at": now.isoformat(timespec="microseconds"),   # sort key: time as text, sorts correctly
        "role": role,                                           # "user" or "assistant"
        "text": text,
        "expires_at": int(now.timestamp()) + 30 * 24 * 3600,    # TTL: Unix seconds, 30 days from now
    })

save_message("S-100", "user", "How many leave days do I have?")
save_message("S-100", "assistant", "You have 16 days.")
save_message("S-100", "user", "Can I carry them over?")
save_message("S-100", "assistant", "Up to 10 days.")
save_message("S-200", "user", "What is the travel policy?")
print("Saved 5 messages in 2 sessions")

## 10. Chat memory: load the last messages of a session

```
 query(session_id = "S-100", newest first, limit N) ──> reverse to oldest first ──> send to the LLM
```

This is exactly what an assistant does before every answer.

In [ ]:
def last_messages(session_id, n):
    response = chats.query(
        KeyConditionExpression=Key("session_id").eq(session_id),
        ScanIndexForward=False,      # newest first
        Limit=n,
    )
    return list(reversed(response["Items"]))   # oldest first, the order an LLM expects

for m in last_messages("S-100", 3):
    print(f'{m["role"]:>9}: {m["text"]}')

Turn them into the message list most LLM APIs (e.g. Amazon Bedrock Converse) expect:

In [ ]:
history = [{"role": m["role"], "content": [{"text": m["text"]}]} for m in last_messages("S-100", 10)]
history

Other sort key questions — messages in a time range, or on one day:

In [ ]:
today = datetime.now(timezone.utc).date().isoformat()          # e.g. "2026-11-01"

response = chats.query(
    KeyConditionExpression=Key("session_id").eq("S-100") & Key("created_at").begins_with(today)
)
print("Messages today in S-100:", response["Count"])

## 11. Clean up

```
 delete_table ──> wait until gone
```

In [ ]:
for table in (employees, chats):
    table.delete()
    table.wait_until_not_exists()
    print("Deleted", table.name)

## Quick reference

| Task | Call |
|---|---|
| Connect | `dynamodb = boto3.resource("dynamodb", region_name="us-east-1")` |
| Create table | `dynamodb.create_table(...)` then `.wait_until_exists()` |
| Use an existing table | `dynamodb.Table("Employees")` |
| Add / replace item | `table.put_item(Item={...})` |
| Add many | `with table.batch_writer() as b: b.put_item(Item=...)` |
| Get one | `table.get_item(Key={...})["Item"]` |
| Update some fields | `table.update_item(Key, UpdateExpression, ExpressionAttributeValues)` |
| Update only if … | add `ConditionExpression` |
| Delete one | `table.delete_item(Key={...})` |
| Query by key | `table.query(KeyConditionExpression=Key("pk").eq(...))` |
| Query an index | add `IndexName="..."` |
| Newest first, last N | `ScanIndexForward=False, Limit=N` |
| Read everything | `table.scan()["Items"]` (small tables only) |
| Delete table | `table.delete()` |

**Common errors**

| Error | Meaning |
|---|---|
| `ResourceNotFoundException` | Wrong table name, or the table is in another Region |
| `ResourceInUseException` on create | The table already exists — use `dynamodb.Table("…")` |
| `ValidationException … key` | Key missing or wrong type (`"E1001"` vs `1001`) |
| `ValidationException … reserved keyword` | Names like `name`, `status`, `date` in an expression — use `#n` + `ExpressionAttributeNames` |
| `TypeError: Float types are not supported` | Write decimals as `Decimal("4.5")` |
| `ConditionalCheckFailedException` | Your condition was false — the safety check worked |